In [0]:
CREATE OR REPLACE TABLE production.supply_analytics.pricing_features_audit_export AS

WITH tour_features AS (
  SELECT 
    tour_id,
    supplier_id,
    MAX(tour_title) as tour_title,
    MAX(location_name) as location_name,
    MAX(activity_type) as activity_type,
    MAX(tour_category) as tour_category,
    MAX(date_of_creation) as date_of_creation,
    MAX(date_of_first_online) as date_of_first_online,
    
    -- API Integration (if ANY option uses it)
    MAX(uses_prices_over_api) as uses_prices_over_api,
    MAX(uses_live_availability_and_price) as uses_live_availability_and_price,
    MAX(uses_pricing_scales_over_api) as uses_pricing_scales_over_api,
    MAX(uses_external_pricing) as uses_external_pricing,
    
    -- Tour structure
    COUNT(DISTINCT tour_option_id) as num_tour_options,
    COUNT(DISTINCT pricing_id) as num_pricing_configs,
    
    -- Feature flags (if ANY option has it)
    MAX(has_addons_configured) as has_addons_configured,
    MAX(has_individual_pricing) as has_individual_pricing,
    MAX(has_group_pricing) as has_group_pricing,
    MAX(has_any_scale_pricing) as has_any_scale_pricing,
    MAX(has_scale_pricing_individual) as has_scale_pricing_individual,
    MAX(has_scale_pricing_group) as has_scale_pricing_group,
    MAX(has_scale_pricing_addons) as has_scale_pricing_addons,
    
    -- Operational features
    MAX(has_cutoff_configured) as has_cutoff_configured,
    MAX(has_min_participant_requirement) as has_min_participant_requirement,
    MAX(has_max_participant_limit) as has_max_participant_limit,
    MAX(has_capacity_limits) as has_capacity_limits,
    MAX(has_available_slots) as has_available_slots,
    
    -- Pricing complexity (averages across options)
    ROUND(AVG(num_individual_categories), 2) as avg_individual_categories,
    ROUND(AVG(num_group_categories), 2) as avg_group_categories,
    ROUND(AVG(num_addons), 2) as avg_addons,
    ROUND(AVG(total_individual_price_tiers), 2) as avg_individual_tiers,
    ROUND(AVG(total_group_price_tiers), 2) as avg_group_tiers,
    ROUND(AVG(total_addon_price_tiers), 2) as avg_addon_tiers,
    ROUND(AVG(total_price_tiers), 2) as avg_total_tiers,
    
    -- Operational metrics
    ROUND(AVG(cutoff_hours), 1) as avg_cutoff_hours,
    ROUND(AVG(min_booking_participants), 1) as avg_min_participants,
    ROUND(AVG(max_booking_participants), 1) as avg_max_participants,
    ROUND(AVG(pricing_dates_covered_next_12mo), 0) as avg_dates_covered,
    ROUND(AVG(pct_dates_with_vacancy_info), 1) as avg_vacancy_coverage_pct,
    
    -- Capacity metrics
    ROUND(AVG(sample_vacancies), 1) as avg_current_vacancies,
    ROUND(AVG(sample_max_vacancies), 1) as avg_max_vacancies,
    
    -- Latest config timestamp
    MAX(config_snapshot_timestamp) as latest_config_update,
    
    -- Feature count (sum of key features)
    (MAX(has_any_scale_pricing) + 
     MAX(has_addons_configured) + 
     MAX(has_group_pricing) + 
     MAX(has_capacity_limits) + 
     MAX(has_cutoff_configured)) as feature_count
    
  FROM production.supply_analytics.feature_audit_base
  GROUP BY tour_id, supplier_id
),

performance_data AS (
  SELECT 
    tour_id,
    supplier_id,
    
    -- L3M metrics
    bookings_l3mo,
    gmv_l3mo,
    nr_l3mo,
    tickets_l3mo,
    customers_l3mo,
    
    -- L6M metrics
    bookings_l6mo,
    gmv_l6mo,
    nr_l6mo,
    tickets_l6mo,
    customers_l6mo,
    
    -- L12M metrics
    bookings_l12mo,
    gmv_l12mo,
    nr_l12mo,
    tickets_l12mo,
    customers_l12mo,
    
    -- Lifetime
    bookings_lifetime,
    gmv_lifetime,
    nr_lifetime,
    
    -- Average metrics
    avg_booking_value_l12mo,
    avg_tickets_per_booking_l12mo,
    
    -- Engagement
    last_booking_date,
    first_booking_date,
    days_since_last_booking,
    booking_growth_trend_pct,
    
    -- Repeat customers
    repeat_customer_rate_l3mo,
    repeat_customer_rate_l6mo,
    repeat_customer_rate_l12mo,
    
    -- Percentiles & tiers
    gmv_percentile,
    bookings_percentile,
    nr_percentile,
    basket_size_percentile,
    gmv_decile,
    gmv_quartile,
    gmv_tier,
    tour_value_tier
    
  FROM production.supply_analytics.feature_performance
)

SELECT 
  -- Tour & Supplier IDs
  tf.tour_id,
  tf.supplier_id,
  
  -- Tour metadata
  tf.tour_title,
  tf.location_name,
  tf.activity_type,
  tf.tour_category,
  tf.date_of_creation,
  tf.date_of_first_online,
  DATEDIFF(CURRENT_DATE, tf.date_of_creation) as tour_age_days,
  DATEDIFF(CURRENT_DATE, tf.date_of_first_online) as days_online,
  
  -- API integration
  tf.uses_prices_over_api,
  tf.uses_live_availability_and_price,
  tf.uses_pricing_scales_over_api,
  tf.uses_external_pricing,
  
  -- Tour structure
  tf.num_tour_options,
  tf.num_pricing_configs,
  
  -- Feature flags
  tf.has_addons_configured,
  tf.has_individual_pricing,
  tf.has_group_pricing,
  tf.has_any_scale_pricing,
  tf.has_scale_pricing_individual,
  tf.has_scale_pricing_group,
  tf.has_scale_pricing_addons,
  tf.has_cutoff_configured,
  tf.has_min_participant_requirement,
  tf.has_max_participant_limit,
  tf.has_capacity_limits,
  tf.has_available_slots,
  tf.feature_count,
  
  -- Pricing complexity
  tf.avg_individual_categories,
  tf.avg_group_categories,
  tf.avg_addons,
  tf.avg_individual_tiers,
  tf.avg_group_tiers,
  tf.avg_addon_tiers,
  tf.avg_total_tiers,
  
  -- Operational config
  tf.avg_cutoff_hours,
  tf.avg_min_participants,
  tf.avg_max_participants,
  tf.avg_dates_covered,
  tf.avg_vacancy_coverage_pct,
  tf.avg_current_vacancies,
  tf.avg_max_vacancies,
  tf.latest_config_update,
  
  -- Performance L3M
  COALESCE(pd.bookings_l3mo, 0) as bookings_l3mo,
  COALESCE(pd.gmv_l3mo, 0) as gmv_l3mo,
  COALESCE(pd.nr_l3mo, 0) as nr_l3mo,
  COALESCE(pd.tickets_l3mo, 0) as tickets_l3mo,
  COALESCE(pd.customers_l3mo, 0) as customers_l3mo,
  
  -- Performance L6M
  COALESCE(pd.bookings_l6mo, 0) as bookings_l6mo,
  COALESCE(pd.gmv_l6mo, 0) as gmv_l6mo,
  COALESCE(pd.nr_l6mo, 0) as nr_l6mo,
  COALESCE(pd.tickets_l6mo, 0) as tickets_l6mo,
  COALESCE(pd.customers_l6mo, 0) as customers_l6mo,
  
  -- Performance L12M
  COALESCE(pd.bookings_l12mo, 0) as bookings_l12mo,
  COALESCE(pd.gmv_l12mo, 0) as gmv_l12mo,
  COALESCE(pd.nr_l12mo, 0) as nr_l12mo,
  COALESCE(pd.tickets_l12mo, 0) as tickets_l12mo,
  COALESCE(pd.customers_l12mo, 0) as customers_l12mo,
  
  -- Lifetime performance
  COALESCE(pd.bookings_lifetime, 0) as bookings_lifetime,
  COALESCE(pd.gmv_lifetime, 0) as gmv_lifetime,
  COALESCE(pd.nr_lifetime, 0) as nr_lifetime,
  
  -- Derived metrics
  pd.avg_booking_value_l12mo,
  pd.avg_tickets_per_booking_l12mo,
  
  CASE 
    WHEN pd.bookings_l12mo > 0 
    THEN ROUND(pd.gmv_l12mo / pd.bookings_l12mo, 2)
    ELSE NULL 
  END as calculated_aov_l12mo,
  
  CASE 
    WHEN pd.customers_l12mo > 0 
    THEN ROUND(pd.bookings_l12mo * 1.0 / pd.customers_l12mo, 2)
    ELSE NULL 
  END as bookings_per_customer_l12mo,
  
  CASE 
    WHEN pd.bookings_l12mo > 0 
    THEN ROUND(pd.tickets_l12mo * 1.0 / pd.bookings_l12mo, 2)
    ELSE NULL 
  END as tickets_per_booking_l12mo,
  
  -- Engagement metrics
  pd.last_booking_date,
  pd.first_booking_date,
  pd.days_since_last_booking,
  pd.booking_growth_trend_pct,
  pd.repeat_customer_rate_l12mo,
  
  -- Tiers & segments
  pd.gmv_percentile,
  pd.bookings_percentile,
  pd.basket_size_percentile,
  pd.gmv_decile,
  pd.gmv_tier,
  pd.tour_value_tier,
  
  -- Derived segments
  CASE 
    WHEN tf.uses_external_pricing = 1 THEN 'external_pricing'
    WHEN pd.tour_value_tier IN ('top_tier', 'high_value') AND tf.feature_count >= 4 THEN 'high_value_full_features'
    WHEN pd.tour_value_tier IN ('top_tier', 'high_value') AND tf.feature_count <= 1 THEN 'high_value_low_features'
    WHEN pd.tour_value_tier = 'medium_value' AND tf.feature_count <= 2 THEN 'medium_value_opportunity'
    WHEN pd.bookings_l12mo = 0 THEN 'no_bookings'
    ELSE 'standard'
  END as tour_segment,
  
  -- Opportunity flags
  CASE 
    WHEN tf.has_any_scale_pricing = 0 
      AND pd.tour_value_tier IN ('top_tier', 'high_value', 'medium_value')
      AND pd.bookings_l12mo >= 30
      AND tf.has_individual_pricing = 1
      AND tf.uses_external_pricing = 0
    THEN 1 ELSE 0 
  END as scale_pricing_opportunity,
  
  CASE 
    WHEN tf.has_addons_configured = 0 
      AND pd.tour_value_tier IN ('top_tier', 'high_value')
      AND pd.bookings_l12mo >= 20
      AND tf.uses_external_pricing = 0
    THEN 1 ELSE 0 
  END as addon_opportunity,
  
  CASE 
    WHEN tf.has_group_pricing = 0 
      AND pd.avg_tickets_per_booking_l12mo >= 3.0
      AND pd.tour_value_tier IN ('top_tier', 'high_value', 'medium_value')
      AND pd.bookings_l12mo >= 20
      AND tf.uses_external_pricing = 0
    THEN 1 ELSE 0 
  END as group_pricing_opportunity,
  
  -- Estimated revenue opportunities (conservative)
  CASE 
    WHEN tf.has_any_scale_pricing = 0 AND pd.gmv_l12mo > 0
    THEN ROUND(pd.gmv_l12mo * 0.15 * 0.5, 2)  -- 15% lift, 50% achievable
    ELSE 0 
  END as est_scale_pricing_uplift_eur,
  
  CASE 
    WHEN tf.has_addons_configured = 0 AND pd.gmv_l12mo > 0
    THEN ROUND(pd.gmv_l12mo * 0.10 * 0.5, 2)  -- 10% lift, 50% achievable
    ELSE 0 
  END as est_addon_uplift_eur,
  
  -- Data quality flags
  CASE 
    WHEN pd.tour_id IS NULL THEN 'no_bookings_data'
    WHEN tf.num_pricing_configs = 0 THEN 'no_pricing_config'
    WHEN tf.avg_dates_covered < 30 THEN 'limited_availability'
    ELSE 'complete'
  END as data_quality

FROM tour_features tf
LEFT JOIN performance_data pd ON tf.tour_id = pd.tour_id
WHERE tf.uses_external_pricing = 0  -- Focus on native pricing
ORDER BY pd.gmv_l12mo DESC NULLS LAST;
